In [34]:
!pip install langchain>=1.0.0
!pip install langchain-core>=1.0.0
!pip install langchain-openai
!pip install langchain langchain-anthropic langgraph
!pip install tiktoken
!pip install langchain-community

In [1]:
!pip install unstructured
!pip install chromadb
!pip install -U sentence-transformers
!pip install langchain-classic

  Using cached chromadb-1.5.7-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached build-1.4.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [35]:
from langchain_openai import ChatOpenAI
import os
import openai
from google.colab import userdata

# API 키 설정하기.
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ChatOpenAI 모델 초기화하기.
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.5
)

In [36]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


In [37]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A,B):
  return dot(A,B) / (norm(A)*norm(B))

In [26]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

embeddings_model = HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device' : 'cpu'},
    encode_kwargs={'normalize_embeddings':True},
    )

/tmp/ipykernel_6699/2974059554.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceBgeEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [38]:
loader = DirectoryLoader(path='/content/sample_data', glob = '*.txt', loader_cls = TextLoader)
#loader = TextLoader('/content/sample_data/dummy_data.txt')
data = loader.load()


for d in data:
  print(d.metadata)

In [39]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 50,
    length_function = len,
)

docs  = text_splitter.split_documents(data)

In [29]:
# 유사도 직접 확인하기 위한 추가 코드
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 50,
    length_function = len,
)

docs  = text_splitter.split_documents(data)

docs_content = list()

for i in range(len(docs)):
  print(docs[i].page_content + "\n-------------------------\n")
  docs_content.append(docs[i].page_content)

embeddings = embeddings_model.embed_documents(docs_content)

embedded_query = embeddings_model.embed_query("알레르기성 비염에 효과 있는 약은 뭐예요?")
# "알레르기성 비염에 효과 있는 약은 뭐예요? -> 세노바퀵연질캡슐 포함된 chunk의 유사도가 가장 높게 나옴."
# "알레르기에 효과 있는 약은 뭐예요? -> 타이레놀 포함된 chunk의 유사도가 더 높게 나옴."
# 즉, 효능에 적혀 있는 단어 자체가 아니면 제대로 유사도를 판단하지 못함.
for embedding in embeddings:
  print(cos_sim(embedding, embedded_query))

In [40]:
vectorstore = Chroma.from_documents(
    docs,
    embeddings_model,
    collection_name = 'medicines',
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space' : 'cosine'},
)

ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

In [174]:
from collections import Counter
from langchain_classic.retrievers import MultiQueryRetriever

query = "두통이 있어. 타이레놀을 먹으면 돼?"
# 제대로 된 답변

query2 = "감기가 있어. 어떤 약을 먹으면 돼?"
# 두통/감기 등 정확한 용어가 아닌 기침을 언급하면 약을 찾지 못함.

query3 = "알레르기 증상이 있어. 어떤 약을 먹으면 돼?"
# 알레르기 / 알러지 / 알레르리성으로 못 찾음.
# 알레르기성 비염 / 결막염 등 정확한 증상을 설명해야 함.

retriever = vectorstore.as_retriever(
    search_kwargs={"k" : 5}
)


template = ''' 다음 context에는 약의 정보인 "효능", "이름"이 포함되어 있습니다.

질문에 주어진 증상과 관련된 "효능"을 가진 약을 찾으세요.

반드시 다음 순서로 생각하세요:
1. 질문에서 증상(예: 두통, 감기 등)을 파악한다.
2. context의 "효능"에서 증상을 찾는다.
3. 그 효능을 가진 약의 "이름"을 찾는다.
4. 만약, 약의 이름을 찾지 못한다면 다른 약을 추천하지 않기.


답변의 조건
- 언급한 증상과 효과적인 약을 언급하기.
- "context" 단어를 포함시키지 않기.
- 그 약이 가진 부가적은 효능, 성분, 특징 등을 부가적으로 추가 설명하기.


context
: {context}

질문
: {question}

답변:
'''

#template2와 search_prompt는 일단 작성함. (사용하지는 않았음.)
template2 = '''
 다음 단어와 유사한 의미를 가지는 단어 10개를 찾으세요 : {word}

'''

prompt = ChatPromptTemplate.from_template(template)
search_prompt = ChatPromptTemplate.from_template(template2)


# 검색한 문서 결과를 한 개의 문단으로 합침.
def format_docs(document):
  return '\n\n'.join(doc.page_content for doc in document)


def get_filtered_docs(x):
  query = x["question"]

  most_docs = vectorstore.similarity_search(query, k =3)
  sources = [d.metadata["source"] for d in most_docs]
  source_file = Counter(sources).most_common(1)[0][0]

  filtered_docs = vectorstore.similarity_search(
      query,
      k=10,
      filter={"source" : source_file}
  )

  return format_docs(filtered_docs)


rag_chain = (
    {"context" : RunnableLambda(get_filtered_docs), "question" : RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

response = rag_chain.invoke({"question" : query3})
print(response)

알레르기 증상에 대한 효능을 가진 약은 찾을 수 없습니다. 타이레놀은 감기로 인한 발열 및 통증, 두통, 신경통, 근육통, 월경통, 염좌통에 효과적이지만, 알레르기 증상에는 적합하지 않습니다. 다른 약을 추천하지 않겠습니다.


In [175]:
# MultiQueryRetriever 코드
from langchain_classic.retrievers import MultiQueryRetriever


# query3으로 "세노바퀵연질캡슐"이 답변되기는 함.
# 이전에 rag_chain만 사용할 때에는 아예 "세노바퀵연질캡슐"이 답변되지 않음.
# 하지만, 실행에 따라서 매번 결과가 바뀜.
# 타이레놀이 나오거나 추천할 약이 없다고 답변할 때도 있음.
# 콧물 + 기침으로는 감기를 유추하지 못함. (알레르기 -> 알레르기성 등으로는 가능)

query4 = "콧물이랑 기침이 나와. 어떤 약을 먹어야 해?"

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = retriever,
    llm = llm
)

relevant_docs = multiquery_retriever.invoke(query3)

context = format_docs(relevant_docs)

response = rag_chain.invoke({"context" : context, "question" : query3})

print(response)

알레르기 증상에 효과적인 약을 찾는다면, "세노바퀵연질캡슐"을 추천합니다. 이 약은 계절성 및 다년성 알레르기성 비염(코염), 알레르기성 결막염, 만성 특발성 두드러기, 그리고 피부 소양증(가려움증) 등의 증상에 효과적입니다. 

세노바퀵의 주성분은 알레르기 증상을 완화하는 데 도움을 주는 성분으로 구성되어 있으며, 일반적으로 알레르기 반응으로 인한 불편함을 줄여줍니다. 이 약은 무색 내지 연보라색의 내용물이 든 보라색의 투명한 타원형 연질캡슐 형태로 제공되어 복용이 용이합니다.
